In [1]:
import pandas as pd

picai_clinical = pd.read_csv('picai_labels/clinical_information/marksheet.csv')
print(picai_clinical.shape)
picai_clinical.head()

(1500, 14)


,patient_id,study_id,mri_date,patient_age,psa,psad,prostate_volume,histopath_type,lesion_PIRADS,lesion_GS,lesion_ISUP,case_ISUP,case_csPCa,center
0,10000,1000000,2019-07-02,73,7.7,NaN,55.0,MRBx,4,0+0,0,0,NO,PCNN
1,10001,1000001,2016-05-27,64,8.7,0.09,102.0,NaN,2,NaN,NaN,0,NO,RUMC
2,10002,1000002,2021-04-18,58,4.2,0.06,74.0,NaN,2,NaN,NaN,0,NO,ZGT
3,10003,1000003,2019-04-05,72,13.0,NaN,71.5,SysBx,2,0+0,0,0,NO,ZGT
4,10004,1000004,2020-10-21,67,8.0,0.10,78.0,SysBx+MRBx,"3,2","0+0,0+0","0,0",0,NO,RUMC


In [2]:
print(picai_clinical.columns.tolist())

['patient_id', 'study_id', 'mri_date', 'patient_age', 'psa', 'psad', 'prostate_volume', 'histopath_type', 'lesion_PIRADS', 'lesion_GS', 'lesion_ISUP', 'case_ISUP', 'case_csPCa', 'center']


In [3]:
print(picai_clinical.shape)
print(picai_clinical['case_csPCa'].value_counts(dropna=False))
print()
print(picai_clinical.isnull().sum())

(1500, 14)
case_csPCa
NO     1075
YES     425
Name: count, dtype: int64

patient_id           0
study_id             0
mri_date             0
patient_age          0
psa                 40
psad               451
prostate_volume     27
histopath_type     468
lesion_PIRADS        0
lesion_GS          468
lesion_ISUP        468
case_ISUP            0
case_csPCa           0
center               0
dtype: int64


In [4]:
picai_clean = picai_clinical.copy()

# Recompute PSAD where missing, using PSA / prostate_volume
picai_clean['psad'] = picai_clean['psad'].fillna(picai_clean['psa'] / picai_clean['prostate_volume'])

# Impute remaining small gaps with median
picai_clean['psa'] = picai_clean['psa'].fillna(picai_clean['psa'].median())
picai_clean['psad'] = picai_clean['psad'].fillna(picai_clean['psad'].median())
picai_clean['prostate_volume'] = picai_clean['prostate_volume'].fillna(picai_clean['prostate_volume'].median())

# Encode label as binary
picai_clean['csPCa_label'] = (picai_clean['case_csPCa'] == 'YES').astype(int)

# Keep only what you need
picai_final = picai_clean[[
    'patient_id', 'study_id', 'patient_age', 'psa', 'psad', 
    'prostate_volume', 'center', 'csPCa_label'
]]

print(picai_final.isnull().sum())
picai_final.head()

patient_id         0
study_id           0
patient_age        0
psa                0
psad               0
prostate_volume    0
center             0
csPCa_label        0
dtype: int64


,patient_id,study_id,patient_age,psa,psad,prostate_volume,center,csPCa_label
0,10000,1000000,73,7.7,0.140000,55.0,PCNN,0
1,10001,1000001,64,8.7,0.090000,102.0,RUMC,0
2,10002,1000002,58,4.2,0.060000,74.0,ZGT,0
3,10003,1000003,72,13.0,0.181818,71.5,ZGT,0
4,10004,1000004,67,8.0,0.100000,78.0,RUMC,0


In [5]:
print(picai_final.isnull().sum())

patient_id         0
study_id           0
patient_age        0
psa                0
psad               0
prostate_volume    0
center             0
csPCa_label        0
dtype: int64


In [6]:
picai_final.to_csv('picai_clinical_clean.csv', index=False)